In [1]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F

from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

from src.utils import get_base_dir

In [8]:
dataset = "dooway"
region = "Basilicata" # "EmiliaRomagna"
region_str = region.lower() # "emilia_romagna"
base_dir = os.path.join(get_base_dir(), region)
nodes_dir = os.path.join(base_dir, "subset", "heterodata", "features")
edges_dir = os.path.join(base_dir, "subset", "heterodata", "edgelists")

NODE FEATURES

In [3]:
def scale_numeric(df, col):
    scaler = MinMaxScaler()
    df[col + '_scaled'] = scaler.fit_transform(df[[col]])
    df.drop([col], axis=1, inplace=True)
    return df

In [4]:
def safe_tensor_conversion(x, dim):
    if isinstance(x, torch.Tensor):
        return torch.tensor(x, dtype=torch.float32)
    """if isinstance(x, str):
        x = np.fromstring(x.strip('[]'), sep=' ')
        return torch.tensor(x, dtype=torch.float32)"""
    if isinstance(x, np.ndarray):
        # Handle NaN arrays explicitly
        if np.isnan(x).all():  # Check if all elements are NaN
            return torch.zeros(dim, dtype=torch.float32)
        else:
            return torch.tensor(x, dtype=torch.float32)
    if isinstance(x, list):
        x = np.array(x, dtype=np.float32)
        if np.isnan(x).all():
            return torch.zeros(dim, dtype=torch.float32)
        return torch.tensor(x, dtype=torch.float32)
    if pd.isna(x):  # Check for scalar NaN
        return torch.zeros(dim, dtype=torch.float32)
    else:
        return torch.zeros(dim, dtype=torch.float32)

In [5]:
def encoding_attributes(df):
    tensors = []
    for col in df.columns:
        print('##### Processing column ', col, ' #####')
        if df[col].dtype == 'int64':
            df = scale_numeric(df, col)
            tensors.append(torch.tensor(df[col+ '_scaled'].values, dtype=torch.int32).unsqueeze(1))
        elif df[col].dtype == 'float64':
            df = scale_numeric(df, col)
            tensors.append(torch.tensor(df[col+ '_scaled'].values, dtype=torch.float32).unsqueeze(1))
        elif df[col].dtype == 'bool':
            tensors.append(torch.tensor(df[col].values, dtype=torch.bool).unsqueeze(1))
        elif df[col].dtype == 'datetime64[ns, UTC]':
            min_date = df[col].min()
            df[col + '_elapsed'] = (df[col] - min_date).dt.total_seconds()
            df.drop([col], axis=1, inplace=True)
            df = scale_numeric(df, col + '_elapsed')
            tensors.append(torch.tensor(df[col + '_elapsed_scaled'].values, dtype=torch.float32).unsqueeze(1))
        elif df[col].dtype == 'object':  # per le colonne con gli embedding
            first_element = df[col].dropna().iloc[0]
            dim = first_element.shape[0]
            #dim = len(first_element) if isinstance(first_element, list) else np.fromstring(first_element.strip('[]'), sep=' ').shape[0]  # str
            embedding_tensors = df[col].apply(
                lambda x: safe_tensor_conversion(x, dim) if not isinstance(x, torch.Tensor) else x)
            embedding_stack = torch.stack(embedding_tensors.tolist())  # Convert to list before stacking
            tensors.append(embedding_stack)
        elif df[col].dtype == 'category':
            enc = LabelEncoder()
            encoded_values = enc.fit_transform(df[col])
            tensors.append(torch.tensor(encoded_values, dtype=torch.int32).unsqueeze(1))
        else:
            raise ValueError(f"Unsupported column type: {df[col].dtype}")

    # Concatenate all tensors along the last dimension
    return torch.cat(tensors, dim=1)

In [6]:
def apply_PCA(tensor, out_dim):
    n_samples, n_features = tensor.shape
    if n_samples < out_dim:
        if n_features >= out_dim:
            return X[:, :out_dim]  # Truncate if already wide
        else:
            return F.pad(X, (0, out_dim - n_features))  # Pad with zeros
    else:
        pca_tensor = PCA(n_components=out_dim).fit(tensor)
        return torch.tensor(pca_tensor.transform(tensor)).float()

In [9]:
node_types = ["category", "image", "poi", "review", "service", "user"]

for ntype in node_types:
    print(f"##### Processing node type {ntype} #####")
    df = pd.read_parquet(os.path.join(nodes_dir, "dataframes", f"{ntype}.parquet"))
    print(f"Shape of the dataframe: {df.shape}")
    print(f"Columns: {df.columns}")
    df.drop(["unique_id"], axis=1, inplace=True)
    X = encoding_attributes(df)
    print(f"Shape of the tensor: {X.shape}")
    torch.save(X, os.path.join(nodes_dir, "tensors", f"{ntype}.pt"))
    

##### Processing node type category #####
Shape of the dataframe: (548, 2)
Columns: Index(['unique_id', 'category_encoded'], dtype='object')
##### Processing column  category_encoded  #####
Shape of the tensor: torch.Size([548, 256])
##### Processing node type image #####
Shape of the dataframe: (4595, 2)
Columns: Index(['unique_id', 'main_image_embedding'], dtype='object')
##### Processing column  main_image_embedding  #####
Shape of the tensor: torch.Size([4595, 512])
##### Processing node type poi #####
Shape of the dataframe: (4595, 26)
Columns: Index(['rank_absolute', 'total_photos', 'latitude', 'longitude', 'is_claimed',
       'unique_id', 'restaurant', 'hotel', 'attraction', 'avg_rating',
       'votes_count', 'inexpensive', 'moderate', 'expensive', 'very_expensive',
       'open24/7', 'title_embedding', 'free_parking', 'paid_parking',
       'reservation', 'excursion', 'spa', 'vegetarian_cuisine',
       'vegan_cuisine', 'gluten-free_cuisine', 'lactose-free_cuisine'],
      dt

In [10]:
# Applying PCA

node_types = ["category", "image", "poi", "review", "service", "user"]
dim = 256

for ntype in node_types:
    print(f"##### Processing node type {ntype} #####")
    X = torch.load(os.path.join(nodes_dir, "tensors", f"{ntype}.pt"))
    print(f"Initial shape: {X.shape}")
    X_pca = apply_PCA(X, dim)
    print(f"Final shape: {X_pca.shape}")
    torch.save(X_pca, os.path.join(nodes_dir, "tensors", f"{ntype}_{dim}.pt"))



##### Processing node type category #####
Initial shape: torch.Size([548, 256])
Final shape: torch.Size([548, 256])
##### Processing node type image #####
Initial shape: torch.Size([4595, 512])
Final shape: torch.Size([4595, 256])
##### Processing node type poi #####
Initial shape: torch.Size([4595, 408])
Final shape: torch.Size([4595, 256])
##### Processing node type review #####
Initial shape: torch.Size([1597215, 776])
Final shape: torch.Size([1597215, 256])
##### Processing node type service #####
Initial shape: torch.Size([6, 384])
Final shape: torch.Size([6, 256])
##### Processing node type user #####
Initial shape: torch.Size([3543, 256])
Final shape: torch.Size([3543, 256])


In [11]:
# Ground truth

gt = pd.read_parquet(os.path.join(base_dir, "subset", "heterodata", "user_labels.parquet"))
gt.tail()


,unique_id_user,label_id
3538,3538,4
3539,3539,9
3540,3540,9
3541,3541,4
3542,3542,11


In [14]:
gt_tensor = torch.tensor(gt["label_id"].values, dtype=torch.long)
torch.save(gt_tensor, os.path.join(base_dir, "subset", "heterodata", "user_labels.pt"))
gt_tensor.shape

torch.Size([3543])

In [15]:
gt_tensor

tensor([11,  9,  9,  ...,  9,  4, 11])

EDGES

In [11]:
def edges_encoding(df):
    return torch.tensor(df.values.T)


def edges_rev_encoding(df):
    df_rev = df[['tgt', 'src']]
    return torch.tensor(df_rev.values.T)


def convert_edges_to_tensors(fname_no_ext):
    df = pd.read_parquet(os.path.join(edges_dir, "dataframes", f"{fname_no_ext}.parquet"))
    df.rename(columns={df.columns[0]: 'src', df.columns[1]: 'tgt'}, inplace=True)
    df = df[["src", "tgt"]].copy()
    tensor = edges_encoding(df)
    tensor_rev = edges_rev_encoding(df)
    torch.save(tensor, os.path.join(edges_dir, "tensors", f"{fname_no_ext}.pt"))
    torch.save(tensor_rev, os.path.join(edges_dir, "tensors", f"{fname_no_ext}_rev.pt"))

In [12]:
directory = os.path.join(edges_dir, "dataframes")
in_fnames = [f for f in os.listdir(directory) if os.path.isfile(os.path.join(directory, f))]
for in_fname in in_fnames:
    fname_no_ext = os.path.splitext(in_fname)[0]
    parts = fname_no_ext.split("_")
    ntype1 = parts[0]
    ntype2 = parts[-1]
    print(f"##### Processing edge type between {ntype1} and {ntype2} #####")
    convert_edges_to_tensors(fname_no_ext)


##### Processing edge type between user and poi #####
##### Processing edge type between review and poi #####
##### Processing edge type between user and review #####
##### Processing edge type between poi and zone #####
##### Processing edge type between poi and category #####
##### Processing edge type between review and service #####
##### Processing edge type between poi and image #####
